<a href="https://colab.research.google.com/github/KN-Vignesh/Projects/blob/main/Titanic_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

!pip install xgboost
from xgboost import XGBClassifier

### Load the train and test datasets

In [33]:
df_train = pd.read_csv('/content/Titanic/train.csv')
df_test = pd.read_csv('/content/Titanic/test.csv')

### Display the first few rows of the training data

In [34]:
df_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Ensure 'Survived' column exists in the test dataframe

In [35]:
if 'Survived' not in df_test.columns:
  df_test['Survived'] = 0

### Define a preprocessing function to clean and transform the data

In [55]:
def Preprocess(df_train, df_test):
  df = pd.concat([df_train, df_test], axis=0)
  df = df.drop(['Name', 'Ticket'], axis=1)
  df['Age'] = df['Age'].fillna(df['Age'].mean())
  df['Cabin'] = df['Cabin'].fillna('A000')
  df['Embarked'] = df['Embarked'].fillna('V')
  df['Fare'] = df['Fare'].fillna(df['Fare'].median())

  df['cabin_Letter'] = df['Cabin'].str.extract(r'([A-Z])')
  df['cabin_Number'] = df['Cabin'].str.extract(r'(\d+)')
  df = df.drop(['Cabin'], axis=1)

  df = pd.get_dummies(df, columns=['cabin_Letter'], prefix= 'cabin')
  df = pd.get_dummies(df, columns=['Embarked'], prefix= 'Embarked')
  df = pd.get_dummies(df, columns=['Sex'], prefix= 'Sex')

  df = df.drop('cabin_T',axis=1)
  df = df.drop('Embarked_V', axis=1)

  df['cabin_Number'] = df['cabin_Number'].fillna(0)
  df['cabin_Number'] = pd.to_numeric(df['cabin_Number'])

  df['PClass_bin_Fare'] = df['Fare'] // df['Pclass']
  df['PClass_bin_Sex'] = df['Pclass'] - df['Sex_female']


  df_train = df.iloc[:len(df_train)]
  df_test = df.iloc[len(df_train):]

  df_test = df_test.drop('Survived', axis=1)

  return df_train, df_test

### Apply the preprocessing function to the datasets

In [56]:
df_train, df_test = Preprocess(df_train, df_test)

### Split the training data into features (X) and target (y) and then into train/test sets

In [57]:
X = df_train.drop('Survived', axis=1)
y = df_train['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
y_train = y_train.values.ravel()

### Check the shapes of the training features and target

In [58]:
X_train.shape, y_train.shape

((712, 21), (712, 1))

### Train a Logistic Regression model

In [59]:
model_1 = LogisticRegression()
model_1.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

### Make predictions using the Logistic Regression model

In [60]:
y_pred = model_1.predict(X_test)

### Calculate the accuracy of the Logistic Regression model

In [62]:
accuracy_score(y_test, y_pred)

0.8268156424581006

### Train an XGBoost Classifier model

In [63]:
model_2 = XGBClassifier()
model_2.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

### Make predictions and calculate accuracy for the XGBoost model

In [64]:
y_pred = model_2.predict(X_test)
accuracy_score(y_test,y_pred)

0.8156424581005587

### Analyze correlations with 'Survived' column in the training data

In [65]:
train_df.corr()['Survived']

,Survived
PassengerId,-0.005007
Survived,1.000000
Pclass,-0.338481
Age,-0.070323
SibSp,-0.035322
Parch,0.081629
Fare,0.257307
cabin_Number,0.229756
cabin_A,-0.318697
cabin_B,0.175095


### Train a Random Forest Classifier model

In [67]:
model_3 = RandomForestClassifier()
model_3.fit(X_train, y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier()

### Make predictions and calculate accuracy for the Random Forest model

In [68]:
y_pred = model_3.predict(X_test)
accuracy_score(y_test, y_pred)

0.8547486033519553

### Generate the submission file using the Random Forest model predictions

In [69]:
pred = model_3.predict(df_test)
final = pd.DataFrame()
final['PassengerId'] = df_test['PassengerId']
final['Survived'] = pred
final.to_csv('submission.csv', index=False)